<a href="https://colab.research.google.com/github/Viresh-tarapur/AIML_and_data_science_projects/blob/main/health1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Data Loading
def load_data(file_path="/content/final_human_health_dataset_updated (1).xlsx"):
    """Loads the dataset from an Excel file."""
    try:
        df = pd.read_excel(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# 2. Data Preprocessing
def preprocess_data(df):
    """Preprocesses the data by handling outliers, encoding, and scaling."""
    # Handle outliers using IQR
    numerical_features = ['Temperature', 'HeartRate', 'SpO2', 'ECG']
    for col in numerical_features:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = np.clip(df[col], lower_bound, upper_bound)

    # Encode target first
    condition_mapping = {'Normal': 0, 'Critical': 1, 'Fever': 2}
    df['Overall_Condition'] = df['Overall_Condition'].map(condition_mapping)

    # One-hot encode categorical condition columns if they exist
    categorical_cols = ['Temperature_Condition', 'HeartRate_Condition', 'SpO2_Condition']
    existing_cats = [col for col in categorical_cols if col in df.columns]
    if existing_cats:
        df = pd.get_dummies(df, columns=existing_cats, drop_first=True)

    # Feature scaling
    scaler = StandardScaler()
    numerical_cols = ['Temperature', 'HeartRate', 'SpO2', 'ECG']
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

    # Feature engineering
    df['HeartRate_Difference'] = df['HeartRate'] - scaler.transform([[0, 70, 0, 0]])[0][1]  # 70 bpm baseline after scaling

    return df, scaler

# 3. Model Training
def train_model(X_train, y_train):
    """Trains a Random Forest model."""
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    return model

# 4. Prediction Function
def predict_condition(input_data, scaler, model):
    """Predicts the condition with fall detection warning."""
    temp = input_data['Temperature']
    hr = input_data['HeartRate']
    spo2 = input_data['SpO2']
    ecg = input_data['ECG']
    fall_detection = input_data['FallDetection']

    if fall_detection == 1:
        print("⚠️ The person has fallen!")

    # Set condition flags
    temp_normal = 1 if temp <= 37.5 else 0
    temp_fever = 1 if 37.5 < temp <= 38 else 0

    hr_normal = 1 if 60 <= hr <= 100 else 0
    hr_fever = 1 if hr > 100 else 0

    spo2_normal = 1 if spo2 >= 95 else 0

    # Prepare DataFrame
    input_df = pd.DataFrame([{
        'Temperature': temp,
        'HeartRate': hr,
        'SpO2': spo2,
        'ECG': ecg,
        'FallDetection': fall_detection,
        'Temperature_Condition_Fever': temp_fever,
        'Temperature_Condition_Normal': temp_normal,
        'HeartRate_Condition_Fever': hr_fever,
        'HeartRate_Condition_Normal': hr_normal,
        'SpO2_Condition_Normal': spo2_normal,
        'HeartRate_Difference': hr - 70
    }])

    # Scale numerics
    input_df[['Temperature', 'HeartRate', 'SpO2', 'ECG']] = scaler.transform(
        input_df[['Temperature', 'HeartRate', 'SpO2', 'ECG']]
    )

    # Align columns with training features (drop any unknown ones)
    expected_features = model.feature_names_in_
    input_df = input_df.reindex(columns=expected_features, fill_value=0)

    # Predict
    prediction = model.predict(input_df)[0]
    condition_labels = {0: 'Normal', 1: 'Critical', 2: 'Fever'}
    return condition_labels[prediction]

# 5. Main Execution
if __name__ == "__main__":
    file_path = "/content/final_human_health_dataset_updated (1).xlsx"
    df = load_data(file_path)

    if df is not None:
        df, scaler = preprocess_data(df)

        # Features and label
        X = df.drop('Overall_Condition', axis=1)
        y = df['Overall_Condition']

        # Drop non-numeric columns
        object_cols = X.select_dtypes(include=['object']).columns
        if len(object_cols) > 0:
            print(f"Dropping non-numeric columns: {list(object_cols)}")
            X = X.drop(columns=object_cols)

        # Split and train
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        model = train_model(X_train, y_train)

        # Predict custom input
        input_data = {
            'Temperature': 29.4,
            'HeartRate': 88,
            'SpO2': 99.7,
            'ECG': 0.82,
            'FallDetection': 1
        }

        prediction = predict_condition(input_data, scaler, model)
        print(f"\nPredicted Overall Condition: {prediction}")

        # Evaluation
        y_pred = model.predict(X_test)
        print("\nModel Evaluation Metrics:")
        print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
        print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
        print(f"Recall   : {recall_score(y_test, y_pred, average='weighted'):.4f}")
        print(f"F1 Score : {f1_score(y_test, y_pred, average='weighted'):.4f}")


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


⚠️ The person has fallen!

Predicted Overall Condition: Normal

Model Evaluation Metrics:
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Data Loading
def load_data(file_path):
    """Loads the dataset from an Excel file."""
    try:
        df = pd.read_excel(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# 2. Data Preprocessing
def preprocess_data(df):
    """Preprocesses the data by handling outliers, scaling numerical features, and one-hot encoding categorical features."""
    # Outlier Handling (using IQR)
    numerical_features = ['Temperature', 'HeartRate', 'SpO2', 'ECG']
    for col in numerical_features:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = np.clip(df[col], lower_bound, upper_bound)

    # One-Hot Encoding
    categorical_cols = ['Temperature_Condition', 'HeartRate_Condition', 'SpO2_Condition']
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    # Feature Scaling
    numerical_cols = ['Temperature', 'HeartRate', 'SpO2', 'ECG']
    scaler = StandardScaler()
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

    # Feature Engineering (Heart Rate Difference)
    df['HeartRate_Difference'] = df['HeartRate'] - scaler.transform([[0, 70, 0, 0]])[0][1]  # baseline 70 bpm

    # Target Encoding
    condition_mapping = {'Normal': 0, 'Critical': 1, 'Fever': 2}
    df['Overall_Condition'] = df['Overall_Condition'].map(condition_mapping)

    return df, scaler

# 3. Model Training
def train_model(X_train, y_train):
    """Trains a RandomForestClassifier on the training data."""
    rf_classifier = RandomForestClassifier(random_state=42)
    rf_classifier.fit(X_train, y_train)
    return rf_classifier

# 4. Prediction Function (includes temperature < 32 override)
def predict_condition(input_data, scaler, rf_classifier):
    """Predicts the Overall_Condition for input data with rule-based override."""
    temp = input_data['Temperature']
    hr = input_data['HeartRate']
    spo2 = input_data['SpO2']
    ecg = input_data['ECG']
    fall_detection = input_data['FallDetection']

    # Fall Detection Alert
    if fall_detection == 1:
        print("⚠️ The person has fallen!")

    # Override Rule: Critical if Temperature < 32
    if temp < 32:
        return 'Critical'

    # Conditions Encoding
    temp_normal = 1 if temp <= 37.5 else 0
    temp_fever = 1 if 37.5 < temp <= 38 else 0
    hr_normal = 1 if 60 <= hr <= 100 else 0
    hr_fever = 1 if hr > 100 else 0
    spo2_normal = 1 if spo2 >= 95 else 0

    # Construct input DataFrame
    input_df = pd.DataFrame([{
        'Temperature': temp,
        'HeartRate': hr,
        'SpO2': spo2,
        'ECG': ecg,
        'FallDetection': fall_detection,
        'Temperature_Condition_Fever': temp_fever,
        'Temperature_Condition_Normal': temp_normal,
        'HeartRate_Condition_Fever': hr_fever,
        'HeartRate_Condition_Normal': hr_normal,
        'SpO2_Condition_Normal': spo2_normal,
        'HeartRate_Difference': hr - 70
    }])

    # Scale input data
    input_df[['Temperature', 'HeartRate', 'SpO2', 'ECG']] = scaler.transform(input_df[['Temperature', 'HeartRate', 'SpO2', 'ECG']])

    # Predict using trained model
    prediction = rf_classifier.predict(input_df)
    condition_mapping = {0: 'Normal', 1: 'Critical', 2: 'Fever'}
    predicted_condition_label = condition_mapping[prediction[0]]
    return predicted_condition_label

# Main Execution
if __name__ == "__main__":
    file_path = "/content/final_human_health_dataset_updated (1).xlsx"  # Update with your correct path
    df = load_data(file_path)

    if df is not None:
        df, scaler = preprocess_data(df)
        X = df.drop('Overall_Condition', axis=1)
        y = df['Overall_Condition']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        rf_classifier = train_model(X_train, y_train)

        # Example Input
        input_data = {
            'Temperature': 37.5,  # Should trigger Critical based on rule
            'HeartRate': 75,
            'SpO2': 95.18,
            'ECG': 0.98,
            'FallDetection': 0
        }

        predicted_condition = predict_condition(input_data, scaler, rf_classifier)
        print(f"Predicted Overall Condition: {predicted_condition}")

        # Model Evaluation
        y_pred = rf_classifier.predict(X_test)
        print("Model Evaluation:")
        print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2f}")
        print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.2f}")
        print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.2f}")
        print(f"F1 Score:  {f1_score(y_test, y_pred, average='weighted'):.2f}")


Predicted Overall Condition: Normal
Model Evaluation:
Accuracy:  1.00
Precision: 1.00
Recall:    1.00
F1 Score:  1.00


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
